<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_5/lessons/lesson_51_ci_cd/note_lesson_51_ci_cd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚙️ Урок 51 — CI/CD (GitHub Actions)

| Крок | Що робимо |
|---|---|
| 0 | workflows курсу як дані (YAML → словник) і пастка з ключем `on` |
| 1 | чи запуститься workflow на цей PR: `branches` і `paths` (вправи 1–2) |
| 2 | порядок jobs за `needs` і скільки чекати на статус (вправи 3–4) |
| 3 | matrix: скільки jobs насправді (вправа 5) |
| 4 | права `GITHUB_TOKEN` у кожного job (вправа 6) |
| 5 | знайди помилку: лінтер workflow (вправа 7) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам.

GitHub Actions — це YAML і кілька простих правил: коли запускати, у якому порядку, з якими правами. Їх відтворимо на Python над справжніми файлами `.github/workflows/` курсу, а поруч — результати їхнього прогону на PR. Повний розбір — у книзі: [Урок 51](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/lesson_50/).

---
## 0. Workflows як дані

In [ ]:
import os
import subprocess

REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"
if not os.path.isdir(".github/workflows"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", ".github/workflows"], check=True)
    os.chdir("course_repo")

from pathlib import Path

import yaml

WORKFLOWS = {path.stem: yaml.safe_load(path.read_text(encoding="utf-8")) for path in Path(".github/workflows").glob("*.yml")}
NEWS = WORKFLOWS["news_hub"]
print(sorted(WORKFLOWS))
print("ключі верхнього рівня news_hub.yml:", list(NEWS))

🔮 **Прогноз:** серед ключів немає `"on"`, зате є `True`. Чому?

<details>
<summary>Відповідь</summary>

PyYAML читає YAML 1.1, де `on`, `yes`, `off` — булеві значення. Ключ `on:` став `True`. GitHub читає цей файл інакше (для нього `on` — рядок), але будь-який скрипт на PyYAML, що аналізує workflows, має це враховувати. Помічник `triggers()` нижче бере обидва варіанти.
</details>

In [ ]:
def triggers(workflow: dict) -> dict:
    """Блок `on:` — під ключем "on" або True (YAML 1.1)."""
    block = workflow.get("on", workflow.get(True))
    return {name: (rule or {}) for name, rule in (block.items() if isinstance(block, dict) else {block: {}}.items())}


print(triggers(NEWS))

---
## 1. Чи запуститься workflow

Для `pull_request` GitHub перевіряє два фільтри:

- `branches` — гілка, **у яку** зливають PR (base). Немає фільтра — будь-яка;
- `paths` — хоч один змінений файл має збігтися з хоч одним шаблоном. Немає фільтра — будь-які файли.

У шаблонах `*` — будь-що в межах однієї папки, `**` — будь-яка кількість папок.

### Вправа 1

`glob_match(path, pattern)` уже написано. Напиши `would_run(workflow, event, base, changed)` → `True`, якщо workflow запуститься на подію `event` (`"pull_request"` чи `"push"`) з базовою гілкою `base` (для push — гілка, куди пушать) і зміненими файлами `changed`.

In [ ]:
import re


def glob_match(path: str, pattern: str) -> bool:
    regex, i = "", 0
    while i < len(pattern):
        if pattern.startswith("**", i):
            regex, i = regex + ".*", i + 2
        elif pattern[i] == "*":
            regex, i = regex + "[^/]*", i + 1
        else:
            regex, i = regex + re.escape(pattern[i]), i + 1
    return re.fullmatch(regex, path) is not None


def would_run(workflow: dict, event: str, base: str, changed: list[str]) -> bool:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    rules = triggers(workflow)
    if event not in rules:
        return False
    rule = rules[event]
    if "branches" in rule and not any(glob_match(base, b) for b in rule["branches"]):
        return False
    if "paths" in rule and not any(glob_match(f, p) for f in changed for p in rule["paths"]):
        return False
    return True
    # END SOLUTION


PR_BASE = "claude/wonderful-ride-cinspw"          # PR уроку 51 стоїть на гілці попереднього PR
CHANGED = ["module_5/lessons/lesson_51_ci_cd/news_hub/scripts/smoke.sh", "docs/modules/m5/lesson_51.md"]
for name in ("news_hub", "crispy_notes", "notebooks"):
    print(f"{name:<13} PR → {would_run(WORKFLOWS[name], 'pull_request', PR_BASE, CHANGED)}")

assert would_run(NEWS, "pull_request", PR_BASE, CHANGED)
assert not would_run(WORKFLOWS["crispy_notes"], "pull_request", PR_BASE, CHANGED)     # файлів нотаток не змінено
assert not would_run(NEWS, "push", "claude/m5-production", CHANGED)                  # push — лише main
assert would_run(NEWS, "push", "main", CHANGED)
print("✅ Вправа 1 пройдена")

### Вправа 2. Workflow книги

Тригери з `django-tests.yml` Django-книги (шляхи переписано на папку проєкту курсу). Збережи в `book_on_our_pr`, чи запустився б він на PR цього уроку, а в `book_on_main_pr` — на такий самий PR у `main`.

🔮 **Прогноз:** що побачить автор PR у першому випадку — червоний хрестик чи щось інше?

In [ ]:
BOOK_WORKFLOW = yaml.safe_load("""
on:
  pull_request:
    branches: [ main, master ]
    paths:
      - 'module_5/lessons/lesson_51_ci_cd/crispy_notes_project/**'
""")
NOTES_CHANGE = ["module_5/lessons/lesson_51_ci_cd/crispy_notes_project/requirements.txt"]

# YOUR CODE HERE
# BEGIN SOLUTION
book_on_our_pr = would_run(BOOK_WORKFLOW, "pull_request", PR_BASE, NOTES_CHANGE)
book_on_main_pr = would_run(BOOK_WORKFLOW, "pull_request", "main", NOTES_CHANGE)
# END SOLUTION

print("PR у гілку попереднього PR:", book_on_our_pr, "| PR у main:", book_on_main_pr)
print("курс:", would_run(WORKFLOWS["crispy_notes"], "pull_request", PR_BASE, NOTES_CHANGE))
assert (book_on_our_pr, book_on_main_pr) == (False, True)
print("✅ Вправа 2 пройдена")

<details>
<summary>Відповідь</summary>

Нічого: ні хрестика, ні галочки — перевірки просто немає. А саме в цьому PR змінено нижню межу simplejwt, яку ловить лише job мінімальних версій. Відсутня перевірка виглядає як успіх — тому в курсі `pull_request` без `branches`.
</details>

---
## 2. Порядок jobs

Jobs без `needs` стартують одночасно. `needs` — ребра графа, як `depends_on` у Compose (урок 50): job стартує, коли всі його `needs` завершились успішно.

### Вправа 3

Напиши `job_waves(workflow)` → список множин job-ів (хвилі запуску). `needs` буває рядком або списком.

In [ ]:
def job_waves(workflow: dict) -> list[set[str]]:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    needs = {}
    for name, job in workflow["jobs"].items():
        value = job.get("needs", [])
        needs[name] = {value} if isinstance(value, str) else set(value)
    waves, done = [], set()
    while len(done) < len(needs):
        ready = {name for name, deps in needs.items() if name not in done and deps <= done}
        if not ready:
            raise ValueError("цикл у needs")
        waves.append(ready)
        done |= ready
    return waves
    # END SOLUTION


news_waves = job_waves(NEWS)
print(news_waves)
assert news_waves == [{"tests", "min-versions", "postgres", "types"}, {"docker"}, {"publish"}]
print("✅ Вправа 3 пройдена")

### Вправа 4. Скільки чекати на статус

Справжні тривалості jobs `news_hub` на PR уроку (секунди, з журналу GitHub). `tests` — matrix з двох jobs, хвиля чекає повільнішого. Порахуй:

- `wall` — час від старту до кінця, якщо хвилі йдуть одна за одною, а jobs у хвилі — паралельно (хвиля триває, скільки найдовший її job);
- `serial` — якби все робив один job по черзі.

`publish` на PR пропускається — 0 с.

In [ ]:
DURATIONS = {"tests": [62, 56], "min-versions": [35], "postgres": [94], "types": [42], "docker": [115], "publish": [0]}

# YOUR CODE HERE
# BEGIN SOLUTION
wall = sum(max(max(DURATIONS[job]) for job in wave) for wave in news_waves)
serial = sum(sum(times) for times in DURATIONS.values())
# END SOLUTION

print(f"паралельно по хвилях: {wall} с ≈ {wall / 60:.1f} хв | по черзі: {serial} с ≈ {serial / 60:.1f} хв")
assert (wall, serial) == (209, 404)
print("✅ Вправа 4 пройдена")

Справжній прогін: jobs стартували о 10:52:23, `docker` — о 10:53:28 (коли закінчився повільніший з `tests`), кінець — 10:55:23: ~3 хвилини — менше за модель; чому — нижче, під діаграмою.

```mermaid
flowchart TD
    classDef step     fill:#263238,stroke:#90a4ae,color:#ffffff;
    classDef decision fill:#37474f,stroke:#64b5f6,color:#ffffff;
    classDef success  fill:#1b5e20,stroke:#4CAF50,color:#ffffff;
    classDef error    fill:#4e1f1f,stroke:#f44336,color:#ffffff;
    classDef warning  fill:#4a3b00,stroke:#ff9800,color:#ffffff;

    subgraph W1["хвиля 1: max = 94 с (postgres)"]
        direction LR
        A["tests 62 / 56"] --> B["min-versions 35"] --> C["postgres 94"] --> D["types 42"]
    end
    subgraph W2["хвиля 2: needs tests → 115 с"]
        direction LR
        E["docker 115"]
    end
    subgraph W3["хвиля 3: needs усі"]
        direction LR
        F{"push у main?"} --> G["publish: skipped"]
    end
    W1 --> W2 --> W3

    class A,B,D success
    class C warning
    class E success
    class F decision
    class G step
```

Хвиля 1 чекає найповільнішого — `postgres`. `docker` залежить лише від `tests`, тож на GitHub він стартує, щойно закінчились обидва `tests` (не чекаючи `postgres`) — справжній прогін ще трохи коротший за нашу модель «хвиль».

---
## 3. Matrix

### Вправа 5

`strategy.matrix` розгортає один job у кілька: декартів добуток усіх списків. Напиши `expand(job)` → список словників-комбінацій (без `matrix` — один порожній словник). Порахуй `total` — скільки jobs насправді запускає `news_hub.yml` на PR (без `publish`).

In [ ]:
import itertools


def expand(job: dict) -> list[dict]:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    matrix = job.get("strategy", {}).get("matrix", {})
    keys = [key for key in matrix if key not in ("include", "exclude")]
    return [dict(zip(keys, values)) for values in itertools.product(*(matrix[key] for key in keys))] or [{}]
    # END SOLUTION


print(expand(NEWS["jobs"]["tests"]), expand(NEWS["jobs"]["types"]))
total = sum(len(expand(job)) for name, job in NEWS["jobs"].items() if name != "publish")
print("jobs на PR:", total)
assert expand(NEWS["jobs"]["tests"]) == [{"python": "3.10"}, {"python": "3.13"}] and total == 6
assert len(expand({"strategy": {"matrix": {"python": ["3.10", "3.13"], "db": ["sqlite", "postgres"]}}})) == 4
print("✅ Вправа 5 пройдена")

---
## 4. Права токена

Кожен job отримує `GITHUB_TOKEN`. Його права: блок `permissions` job-а, якщо є; інакше — блок workflow; інакше — налаштування репозиторію (у старших репозиторіях — запис).

### Вправа 6

Напиши `effective_permissions(workflow, job_name)` → словник прав або рядок `"налаштування репозиторію"`. Потім збери `writers` — множину `(workflow, job)` з будь-яким правом `write` серед workflows `news_hub` і `crispy_notes`.

In [ ]:
def effective_permissions(workflow: dict, job_name: str) -> dict | str:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    job = workflow["jobs"][job_name]
    return job.get("permissions", workflow.get("permissions", "налаштування репозиторію"))
    # END SOLUTION


# YOUR CODE HERE
# BEGIN SOLUTION
writers = {(wf, job) for wf in ("news_hub", "crispy_notes") for job in WORKFLOWS[wf]["jobs"]
           if isinstance(p := effective_permissions(WORKFLOWS[wf], job), dict) and "write" in p.values()}
# END SOLUTION

for job in NEWS["jobs"]:
    print(f"{job:<13} {effective_permissions(NEWS, job)}")
print("пишуть:", writers)
assert writers == {("news_hub", "publish"), ("crispy_notes", "publish")}
assert effective_permissions(yaml.safe_load("jobs: {test: {}}"), "test") == "налаштування репозиторію"
print("✅ Вправа 6 пройдена")

---
## 5. Знайди помилку

Workflow зелений на кожному PR. Проблем у ньому чотири.

### Вправа 7

Напиши `lint(workflow)` → множину кодів:

- `"swallowed-exit"` — у `run` є `|| true` (код виходу проковтнуто — крок завжди зелений);
- `"secret-in-file"` — у `env` (workflow чи job) значення, схоже на ключ: ім'я з `KEY`, `SECRET`, `TOKEN`, `PASSWORD` і значення **не** з `${{ secrets.… }}`;
- `"push-everywhere"` — `push` без `branches`;
- `"pr-only-main"` — `pull_request` з `branches`.

Workflows `news_hub` і `crispy_notes` курсу мають пройти без зауважень (їхні `POSTGRES_PASSWORD` у `services` — у `env` сервісу, не job-а, і `lint` їх не розглядає).

In [ ]:
FIND_BUG = yaml.safe_load("""
on:
  push:
  pull_request:
    branches: [main]
jobs:
  test:
    runs-on: ubuntu-latest
    env:
      GEMINI_API_KEY: AIzaSyD-course-demo-key-not-real-0000000
    steps:
      - uses: actions/checkout@v5
      - uses: actions/setup-python@v6
        with: {python-version: "3.13"}
      - run: pip install -r module_5/lessons/lesson_51_ci_cd/news_hub/requirements.txt
      - run: cd module_5/lessons/lesson_51_ci_cd/news_hub && pytest || true
""")
SECRET_WORDS = ("KEY", "SECRET", "TOKEN", "PASSWORD")


def lint(workflow: dict) -> set[str]:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    problems = set()
    rules = triggers(workflow)
    if "push" in rules and "branches" not in rules["push"]:
        problems.add("push-everywhere")
    if "branches" in rules.get("pull_request", {}):
        problems.add("pr-only-main")
    envs = [workflow.get("env", {})] + [job.get("env", {}) for job in workflow["jobs"].values()]
    for env in envs:
        for name, value in env.items():
            if any(word in name.upper() for word in SECRET_WORDS) and "secrets." not in str(value):
                problems.add("secret-in-file")
    for job in workflow["jobs"].values():
        for step in job.get("steps", []):
            if "|| true" in step.get("run", ""):
                problems.add("swallowed-exit")
    return problems
    # END SOLUTION


print("find-bug:", sorted(lint(FIND_BUG)))
print("курс:", {name: sorted(lint(WORKFLOWS[name])) for name in ("news_hub", "crispy_notes")})
assert lint(FIND_BUG) == {"swallowed-exit", "secret-in-file", "push-everywhere", "pr-only-main"}
assert lint(WORKFLOWS["news_hub"]) == set() and lint(WORKFLOWS["crispy_notes"]) == set()
print("✅ Вправа 7 пройдена")

<details>
<summary>Відповідь</summary>

1. `pytest || true` — крок зелений за будь-якого результату. А ще встановлено `requirements.txt` без pytest: `pytest: command not found`, і `|| true` ховає й це.
2. Ключ у файлі — у git назавжди. Секрети — у Settings → Secrets і `${{ secrets.… }}`; тестам ключ не потрібен (`FakeLLM`).
3. `push` без гілок — прогін на кожен push у кожну гілку (разом з PR — двічі на коміт).
4. `pull_request: branches: [main]` — PR в інші гілки без перевірки.

`smoke.sh` курсу через `trap cleanup EXIT` теж завжди «прибирає за собою», але код виходу не ковтає: обробник без `exit` не змінює код, з яким завершився скрипт (невдалий вхід адміна → код 1, і job червоний). Прибирання й чесний результат не суперечать одне одному.
</details>

---
## Самоперевірка

1. Чому `pull_request` у курсі — без `branches`?
2. Що перевіряє job з `uv --resolution lowest-direct`?
3. Навіщо `needs: tests` у job з образом?
4. Звідки в job `publish` право `packages: write` і чому не в інших?
5. Чим небезпечний зелений CI з `|| true`?

## Далі

- Книга курсу: [Урок 51](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/lesson_50/) — workflow книги проти курсу, справжні прогони на PR, smoke-тести, публікація образу.
- У репозиторії: `.github/workflows/news_hub.yml`, `crispy_notes.yml`; локально — `pytest`, `mypy`, `pytest -m docker`, `./scripts/smoke.sh`.